# Qwen-VL 解析 PDF 首页（Notebook 版）

这个 Notebook 按照 `qwen_vl_pdf_first_page.py` 的逻辑整理而成：
1. 把本地 PDF 的第一页渲染成 PNG
2. 可选地只保存图片，不调用模型
3. 通过 OpenAI 兼容接口调用 Qwen 视觉模型解析首页内容


## 使用说明

- 默认读取环境变量 `DASHSCOPE_API_KEY`
- 默认示例 PDF 为当前目录下的 `1.pdf`
- 如果你只想导出首页图片，把 `SAVE_IMAGE_ONLY` 设为 `True`


In [1]:
from __future__ import annotations

import base64
import mimetypes
import os
import tempfile
from pathlib import Path
from typing import Optional

import pdfplumber
from IPython.display import Image, display


## 1. 配置参数

你可以直接修改下面这些参数，然后顺序执行后续单元。


In [2]:
DEFAULT_BASE_URL = 'https://dashscope.aliyuncs.com/compatible-mode/v1'
DEFAULT_PROMPT = '请解析这页 PDF 的主要内容，提取标题、关键信息，并按条目输出。'
DEFAULT_MODEL = 'qwen3.5-plus'

BASE_DIR = Path.cwd()
PDF_PATH = BASE_DIR / '1.pdf'
OUTPUT_DIR = BASE_DIR / 'tmp_pdf_images'
PROMPT = DEFAULT_PROMPT
MODEL = DEFAULT_MODEL
BASE_URL = DEFAULT_BASE_URL
API_KEY = os.getenv('DASHSCOPE_API_KEY')
RESOLUTION = 160
SAVE_IMAGE_ONLY = False

print('PDF_PATH =', PDF_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('MODEL =', MODEL)
print('BASE_URL =', BASE_URL)
print('RESOLUTION =', RESOLUTION)
print('SAVE_IMAGE_ONLY =', SAVE_IMAGE_ONLY)
print('API_KEY 已设置 =', bool(API_KEY))


PDF_PATH = /Users/xinjianrun/Desktop/架构师资料包/AI大模型课程/第10周：多模态大模型/Week10/1.pdf
OUTPUT_DIR = /Users/xinjianrun/Desktop/架构师资料包/AI大模型课程/第10周：多模态大模型/Week10/tmp_pdf_images
MODEL = qwen3.5-plus
BASE_URL = https://dashscope.aliyuncs.com/compatible-mode/v1
RESOLUTION = 160
SAVE_IMAGE_ONLY = False
API_KEY 已设置 = True


## 2. PDF 首页转图片


In [3]:
def render_first_page_to_png(pdf_path: Path, output_dir: Optional[Path], resolution: int) -> Path:
    if not pdf_path.exists():
        raise FileNotFoundError(f'PDF 文件不存在: {pdf_path}')

    if output_dir is None:
        output_dir = Path(tempfile.mkdtemp(prefix='pdf_page1_'))
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / f'{pdf_path.stem}_page1.png'

    with pdfplumber.open(str(pdf_path)) as pdf:
        if not pdf.pages:
            raise ValueError(f'PDF 没有页面: {pdf_path}')
        first_page = pdf.pages[0]
        first_page_image = first_page.to_image(resolution=resolution)
        first_page_image.save(str(output_path), format='PNG')

    return output_path


## 3. 构造图片 Data URL


In [4]:
def build_data_url(image_path: Path) -> str:
    mime_type, _ = mimetypes.guess_type(str(image_path))
    if not mime_type:
        mime_type = 'image/png'
    image_base64 = base64.b64encode(image_path.read_bytes()).decode('utf-8')
    return f'data:{mime_type};base64,{image_base64}'


## 4. 调用 Qwen-VL

这个单元会保持和脚本版本一致的流式输出方式，同时把结果收集到字典中，方便你后续复用。


In [ ]:
def call_qwen_vl(image_path: Path, prompt: str, model: str, base_url: str, api_key: str):
    from openai import OpenAI

    if not api_key:
        raise ValueError('未提供 API Key，请设置 DASHSCOPE_API_KEY 或在参数中传入。')

    client = OpenAI(api_key=api_key, base_url=base_url)
    image_data_url = build_data_url(image_path)

    completion = client.chat.completions.create(
        model=model,
        messages=[
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'image_url',
                        'image_url': {'url': image_data_url},
                    },
                    {'type': 'text', 'text': prompt},
                ],
            }
        ],
        stream=True,
        extra_body={'enable_thinking': True},
    )

    is_answering = False
    reasoning_parts = []
    answer_parts = []
    usage_info = None

    print('\n' + '=' * 20 + '思考过程' + '=' * 20 + '\n')

    for chunk in completion:
        if not chunk.choices:
            if getattr(chunk, 'usage', None) is not None:
                usage_info = chunk.usage
                print('\nUsage:')
                print(chunk.usage)
            continue

        delta = chunk.choices[0].delta
        reasoning_content = getattr(delta, 'reasoning_content', None)
        if reasoning_content:
            reasoning_parts.append(reasoning_content)
            print(reasoning_content, end='', flush=True)
            continue

        if delta.content and not is_answering:
            print('\n' + '=' * 20 + '完整回复' + '=' * 20 + '\n')
            is_answering = True

        if delta.content:
            answer_parts.append(delta.content)
            print(delta.content, end='', flush=True)

    print()
    return {
        'reasoning': ''.join(reasoning_parts),
        'answer': ''.join(answer_parts),
        'usage': usage_info,
    }


## 5. 渲染并预览首页图片


In [ ]:
page1_image_path = render_first_page_to_png(PDF_PATH, OUTPUT_DIR, RESOLUTION)
print(f'首页图片已生成: {page1_image_path}')
display(Image(filename=str(page1_image_path)))


## 6. 解析首页内容

如果只想导出图片，不调用模型，请把前面的 `SAVE_IMAGE_ONLY` 设为 `True`。


In [ ]:
if SAVE_IMAGE_ONLY:
    print('已设置 SAVE_IMAGE_ONLY=True，跳过模型调用。')
    result = None
else:
    result = call_qwen_vl(
        image_path=page1_image_path,
        prompt=PROMPT,
        model=MODEL,
        base_url=BASE_URL,
        api_key=API_KEY,
    )

result


## 7. 可选：修改提示词后重新调用

如果你想做更结构化的信息抽取，可以在下面这个单元中改提示词后再次执行。


In [ ]:
CUSTOM_PROMPT = '''
请解析这页 PDF，按以下结构输出：
1. 页面标题
2. 文档类型判断
3. 关键信息摘要
4. 如果页面中有表格、编号、时间、金额，请单独列出
5. 如果有 OCR 不确定的地方，请明确说明
'''.strip()

print(CUSTOM_PROMPT)


In [ ]:
if not SAVE_IMAGE_ONLY:
    custom_result = call_qwen_vl(
        image_path=page1_image_path,
        prompt=CUSTOM_PROMPT,
        model=MODEL,
        base_url=BASE_URL,
        api_key=API_KEY,
    )
    custom_result
else:
    print('SAVE_IMAGE_ONLY=True，跳过自定义提示词调用。')
